# Code mẫu — Autoencoder

Autoencoder học hai việc cùng lúc: nén dữ liệu xuống một mã ngắn, rồi dựng lại dữ liệu từ mã ấy. Thứ được tối ưu là reconstruction loss, tức độ lệch giữa bản dựng lại và bản gốc. Nút thắt quá hẹp sẽ bỏ mất những chi tiết cần giữ. Nút thắt quá rộng thì có thể chép nguyên mọi thứ, kể cả những mẫu lạ. Các ví dụ dưới đây luôn có dữ liệu sạch, một baseline để so, và chỉ rõ chỗ mà điểm tái tạo không đáng tin.

## Ví dụ 1: Tính tay một nút thắt

Giả sử dữ liệu bình thường nằm gần đường `x2 = x1`. Encoder lấy trung bình hai tọa độ làm mã. Decoder lặp lại mã ấy cho cả hai tọa độ. Lỗi tái tạo khi đó chỉ đo độ lệch khỏi đường chung, không đo điểm nằm xa hay gần dọc theo đường.

In [ ]:
import numpy as np

sample = np.array([0.9, 0.1])
code = sample.mean()
reconstructed = np.array([code, code])
mse = np.mean((reconstructed - sample) ** 2)
large_on_line = np.array([100., 100.])
large_code = large_on_line.mean()
large_reconstruction = np.array([large_code, large_code])
print("sample -> code -> reconstruction:", sample.tolist(), round(float(code), 4), reconstructed.tolist())
print("reconstruction MSE:", round(float(mse), 4))
print("large sample MSE:", round(float(np.mean((large_reconstruction - large_on_line) ** 2)), 4))
assert np.allclose(reconstructed, [0.5, 0.5])
assert np.isclose(mse, 0.16)
assert np.allclose(large_reconstruction, large_on_line)

sample -> code -> reconstruction: [0.9, 0.1] 0.5 [0.5, 0.5]
reconstruction MSE: 0.16
large sample MSE: 0.0


Mẫu `[0.9, 0.1]` có mã 0.5, và được dựng lại thành `[0.5, 0.5]`. Mỗi tọa độ lệch 0.4, nên MSE bằng 0.16. Mẫu `[100, 100]` thì được dựng lại hoàn hảo, MSE bằng 0, dù nó lớn bất thường. Lý do là nó vẫn nằm đúng trên đường `x2 = x1`. Một mẫu chỉ bất thường về độ lớn, mà vẫn theo đúng cấu trúc chung, sẽ lọt qua cách dò này.

## Ví dụ 2: So autoencoder tuyến tính với PCA

Bốn cảm biến cùng phụ thuộc vào một biến ẩn `t`, cộng thêm chút nhiễu. Ta tách validation trước, rồi mới học mean và std từ train để chuẩn hóa. PCA một chiều, tính bằng SVD, làm baseline. Autoencoder `4 → 1 → 4` được tối ưu theo cùng loss MSE. Ta so hai bên theo lỗi tái tạo. Không cần mã của autoencoder trùng với tọa độ PCA.

In [ ]:
import numpy as np
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(3102)
generator = torch.Generator().manual_seed(3102)
direction = torch.tensor([1., 2., -1., 0.5])

def make_normal(size):
    t = torch.randn(size, 1, generator=generator)
    clean = t * direction
    noisy = clean + 0.03 * torch.randn(size, 4, generator=generator)
    return noisy

raw_train = make_normal(64)
raw_valid = make_normal(24)
mean = raw_train.mean(dim=0)
scale = raw_train.std(dim=0, unbiased=False).clamp_min(1e-6)
train = (raw_train - mean) / scale
valid = (raw_valid - mean) / scale
print("train/valid shapes:", tuple(train.shape), tuple(valid.shape))

train_np = train.numpy()
_, singular_values, vt = np.linalg.svd(train_np, full_matrices=False)
axis = vt[0]
pca_latent = train_np @ axis
pca_reconstruction = pca_latent[:, None] * axis[None, :]
pca_mse = np.mean((pca_reconstruction - train_np) ** 2)
print("PCA train MSE:", round(float(pca_mse), 6))
print("PCA variance fraction:", round(float(
    singular_values[0] ** 2 / np.sum(singular_values ** 2)
), 6))

train/valid shapes: (64, 4) (24, 4)
PCA train MSE: 0.000809
PCA variance fraction: 0.999191


Một chiều PCA giữ 99.92% phương sai, và lỗi tái tạo trên train chỉ khoảng 0.0008. PCA tính ra con số này bằng đại số, không cần train gì. Đó là mốc để so khi ta bắt đầu tối ưu mạng. Bước sau dùng đúng cách chia và cách chuẩn hóa này.

In [ ]:
class LinearAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Linear(4, 1, bias=False)
        self.decoder = nn.Linear(1, 4, bias=False)

    def forward(self, x):
        return self.decoder(self.encoder(x))

autoencoder = LinearAutoencoder()
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=0.04)
initial = (autoencoder(train) - train).square().mean().item()
for _ in range(180):
    optimizer.zero_grad(set_to_none=True)
    reconstruction = autoencoder(train)
    loss = (reconstruction - train).square().mean()
    assert torch.isfinite(loss)
    loss.backward()
    optimizer.step()
autoencoder.eval()
with torch.no_grad():
    final = (autoencoder(train) - train).square().mean().item()
    valid_loss = (autoencoder(valid) - valid).square().mean().item()
    latent_shape = tuple(autoencoder.encoder(valid).shape)
print("autoencoder train MSE:", round(initial, 6), "->", round(final, 6))
print("autoencoder validation MSE:", round(valid_loss, 6))
print("latent shape:", latent_shape)
assert latent_shape == (24, 1)
assert final < initial

autoencoder train MSE: 1.193978 -> 0.001576
autoencoder validation MSE: 0.001727
latent shape: (24, 1)


Sau 180 bước, lỗi tái tạo giảm từ khoảng 1.19 xuống 0.0016. Con số này cùng cỡ với PCA nhưng vẫn cao hơn gấp đôi, vì 180 bước Adam chưa đi hết tới nghiệm tối ưu mà SVD tìm được ngay. Autoencoder tuyến tính và PCA cùng tìm ra một hướng chính của dữ liệu. Lỗi trên validation là 0.0017, sát với train, nên phép nén này dùng tốt cho mẫu mới cùng kiểu. Mã một chiều không tự mang một ý nghĩa hay nhãn cố định nào.

## Ví dụ 3: Nút thắt hẹp và rộng khi dùng để dò bất thường

Ta train một autoencoder có mã một chiều trên dữ liệu bình thường. Để đối chứng, ta dựng thêm một autoencoder có mã bốn chiều, với encoder và decoder đều là ma trận đơn vị. Bản rộng này chép y nguyên mọi đầu vào. Ta cố tình dùng nó làm phản ví dụ cho ý tưởng “lỗi tái tạo cao nghĩa là bất thường”. So sánh ở đây là về sức chứa của nút thắt, chứ không phải hai model được train công bằng như nhau.

In [ ]:
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(3103)
direction = torch.tensor([1., 2., -1., 0.5])

def make_normal(size, seed):
    generator = torch.Generator().manual_seed(seed)
    t = torch.linspace(-1.5, 1.5, size).unsqueeze(1)
    clean = t * direction
    return clean + 0.03 * torch.randn(size, 4, generator=generator)

normal_train = make_normal(48, 1)
normal_valid = make_normal(16, 2)
normal_test = make_normal(16, 3)
anomaly_test = make_normal(16, 4)
anomaly_test[:, 1] += 1.5
print("normal/anomaly shapes:", tuple(normal_test.shape), tuple(anomaly_test.shape))

class BottleneckAutoencoder(nn.Module):
    def __init__(self, latent_width):
        super().__init__()
        self.encoder = nn.Linear(4, latent_width, bias=False)
        self.decoder = nn.Linear(latent_width, 4, bias=False)

    def forward(self, x):
        return self.decoder(self.encoder(x))

def train_autoencoder(model, data, steps, learning_rate):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    for _ in range(steps):
        optimizer.zero_grad(set_to_none=True)
        loss = (model(data) - data).square().mean()
        loss.backward()
        optimizer.step()
    return model.eval()

narrow = train_autoencoder(BottleneckAutoencoder(1), normal_train, 220, 0.04)
wide = BottleneckAutoencoder(4)
with torch.no_grad():
    wide.encoder.weight.copy_(torch.eye(4))
    wide.decoder.weight.copy_(torch.eye(4))
wide.eval()
with torch.no_grad():
    narrow_train_mse = (narrow(normal_train) - normal_train).square().mean().item()
    identity_check = torch.allclose(wide(anomaly_test), anomaly_test)
print("narrow train MSE:", round(narrow_train_mse, 6))
print("wide identity reconstructs anomaly exactly:", identity_check)
assert identity_check

normal/anomaly shapes: (16, 4) (16, 4)
narrow train MSE: 0.000997
wide identity reconstructs anomaly exactly: True


Mẫu bất thường được tạo bằng cách cộng thêm 1.5 vào cảm biến thứ hai, nên nó lệch khỏi hướng chung của dữ liệu. Bản rộng chép y nguyên cả mẫu bất thường này. Lỗi tái tạo của nó bằng 0 với mọi đầu vào.

Giờ ta dùng lỗi tái tạo của từng mẫu làm điểm bất thường. Ngưỡng báo động là phân vị 90 của điểm trên validation bình thường. Mẫu nào vượt ngưỡng thì bị báo là bất thường.

In [ ]:
def score(model, data):
    with torch.no_grad():
        return (model(data) - data).square().mean(dim=1)

for name, model in [("latent=1", narrow), ("latent=4", wide)]:
    valid_score = score(model, normal_valid)
    normal_score = score(model, normal_test)
    anomaly_score = score(model, anomaly_test)
    threshold = torch.quantile(valid_score, 0.9)
    normal_flagged = int((normal_score > threshold).sum())
    flagged = int((anomaly_score > threshold).sum())
    print(name, "valid p90:", round(float(threshold), 6),
          "normal median:", round(float(normal_score.median()), 6),
          "anomaly median:", round(float(anomaly_score.median()), 6),
          "normal flagged:", normal_flagged, "/", len(normal_score),
          "anomaly flagged:", flagged, "/", len(anomaly_score))
    assert torch.isfinite(valid_score).all()

latent=1 valid p90: 0.002056 normal median: 0.000425 anomaly median: 0.412712 normal flagged: 4 / 16 anomaly flagged: 16 / 16
latent=4 valid p90: 0.0 normal median: 0.0 anomaly median: 0.0 normal flagged: 0 / 16 anomaly flagged: 0 / 16


Với mã một chiều, điểm trung vị của mẫu bất thường là 0.41, gấp gần nghìn lần mẫu bình thường. Cả 16 mẫu bất thường đều bị bắt. Nhưng nó cũng báo nhầm 4 trong 16 mẫu bình thường. Bản rộng thì cho điểm 0 với mọi mẫu, nên không bắt được mẫu bất thường nào.

Bài học là nút thắt phải đủ hẹp để chỉ giữ được cấu trúc chung, và khi ấy vẫn cần chọn ngưỡng cẩn thận. Phân vị 90 tính trên chỉ 16 mẫu validation không bảo đảm tỉ lệ báo nhầm sẽ là 10% trên dữ liệu mới. Ở đây tỉ lệ báo nhầm thực tế là 4/16, tức 25%. Khi đánh giá một bộ dò bất thường, phải đọc cả hai con số: bao nhiêu mẫu bình thường bị báo nhầm, và bao nhiêu mẫu bất thường bị bỏ sót.